In [1]:
import pandas as pd
from datetime import datetime, timedelta
from bs4 import BeautifulSoup as bs
import requests
from tqdm import tqdm

In [2]:
numdays = 200

base = datetime.today()
date_list = [base - timedelta(days=x) for x in range(numdays)]

fir = ['team','elo_(k=32)','elo_(k=64)','glicko_1','glicko_2']
sec = ['', 'avg_7d', 'δ7d', 'avg_30d', 'δ30d', 'rating', 'μ', 'σ', 'δrat.7d']

In [3]:
elo_df = pd.DataFrame()

for i in tqdm(date_list,total=len(date_list)):
    date = f"{i.day}-{i.month}-{i.year}"
    url = f"https://www.datdota.com/ratings?date={date}"
    page = requests.get(url)
    
    soup = bs(page.text,'html.parser')
    table_body = soup.find('table')
    
    row_data = []
    for row in table_body.find_all('tr'):
        col = row.find_all('td')
        col = [ele.text.strip() for ele in col]
        row_data.append(col)
    
    idx = pd.MultiIndex(levels=[fir,sec],codes=[[0,1,1,1,1,1,2,2,2,2,2,3,3,3,3,4,4,4,4],[0,5,1,2,3,4,5,1,2,3,4,5,6,7,8,5,6,7,8]], names=['lvl1','lvl2'])
    
    df = pd.DataFrame(row_data[2:],columns=idx)
    
    df['team'] = df['team'].apply(lambda x: x.split('\n')[0])
    df.insert(loc=1,column='date',value=date)
    #df['date'] = date
    
    elo_df = pd.concat([elo_df,df]).reset_index(drop=True)

elo_df.columns = ['_'.join(x) for x in elo_df.columns.to_flat_index()]

100%|████████████████████████████████████████████████████████████████████████████████| 200/200 [03:03<00:00,  1.09it/s]


In [4]:
elo_df.rename({'team_'},inplace=True)

,team_,date_,elo_(k=32)_rating,elo_(k=32)_avg_7d,elo_(k=32)_δ7d,elo_(k=32)_avg_30d,elo_(k=32)_δ30d,elo_(k=64)_rating,elo_(k=64)_avg_7d,elo_(k=64)_δ7d,elo_(k=64)_avg_30d,elo_(k=64)_δ30d,glicko_1_rating,glicko_1_μ,glicko_1_σ,glicko_1_δrat.7d,glicko_2_rating,glicko_2_μ,glicko_2_σ,glicko_2_δrat.7d
0,OG,15-7-2022,1145.78,1149.35,-7.19,1151.75,+17.49,1267.11,1273.66,-13.18,1282.74,+7.32,1921.61,2134.13,85.01,-85.63,1956.03,2076.13,48.04,-2.95
1,Tundra Esports,15-7-2022,1158.21,1168.92,+8.54,1124.08,+46.70,1236.96,1259.57,+13.36,1177.67,+85.27,2007.23,2209.32,80.84,+27.24,1950.72,2068.25,47.01,-2.96
2,PSG.LGD,15-7-2022,1187.48,1161.15,+43.06,1152.81,+67.27,1295.43,1230.64,+103.44,1213.00,+137.56,1803.70,2124.61,128.36,-69.95,1942.89,2075.11,52.89,-2.73
3,Team Aster,15-7-2022,1093.18,1100.74,+2.80,1090.68,+17.34,1121.69,1145.63,-5.66,1130.15,+3.34,1941.14,2254.68,125.42,-72.07,1912.46,2036.00,49.41,-2.87
4,Royal Never Give Up,15-7-2022,1093.16,1061.40,+70.73,1065.09,-12.01,1175.19,1091.49,+198.37,1066.02,+45.09,1948.97,2251.20,120.89,-75.64,1903.34,2028.39,50.02,-2.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17821,Gorillaz-Pride,28-12-2021,1031.99,1032.78,-1.58,1014.93,+40.41,1057.38,1058.78,-2.83,1030.97,+67.47,1177.53,1605.83,171.32,-49.56,1351.09,1525.33,69.70,-1.96
17822,Inverse,28-12-2021,913.36,911.24,+4.28,954.72,-71.31,859.48,856.04,+6.93,926.77,-121.97,790.74,1257.63,186.76,-45.01,1187.90,1383.47,78.23,-1.75
17823,Electronic Boys,28-12-2021,965.76,965.44,-2.44,965.42,-6.76,978.51,977.70,+3.39,958.27,+21.55,1045.54,1432.26,154.69,-55.73,1137.95,1330.41,76.98,-1.77
17824,felt,28-12-2021,957.97,956.94,+2.07,966.33,-18.55,932.47,930.81,+3.33,950.82,-38.68,901.93,1343.58,176.66,-47.88,1130.79,1312.91,72.85,-1.87


In [5]:
elo_df.to_csv('ELO.csv',index=False)